In [1]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:23<00:00, 840.68it/s]


In [4]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:13<00:00, 13073.24it/s]


In [5]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:08<00:00, 20049.94it/s]


In [6]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.50, b=0.75):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [7]:
query = 'will SARS-CoV2 infected people develop immunity'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 24808/24808 [00:00<00:00, 285422.76it/s]


[('azk7dfub', 14.301918635645642),
 ('96p0ktoz', 13.639905395073454),
 ('066rysjh', 13.060805192481691),
 ('jfe8neec', 12.700560840925062),
 ('342thf3o', 12.681350984557405),
 ('svwkcuxf', 12.533966734397683),
 ('eo4ehcjv', 12.060711181423702),
 ('xeyggg1b', 11.944403316673494),
 ('6x1704l3', 11.944403316673494),
 ('lvcp8imi', 11.861606358285838),
 ('9807pgmy', 11.852750794735828),
 ('88px7oq2', 11.746155173861606),
 ('w1l8h8g8', 11.522913632191436),
 ('kn0njkqg', 11.396030256791752),
 ('qbr3aomm', 11.371920616211714),
 ('kmkeetl3', 11.253908371561675),
 ('dlgvge3s', 11.156181248328554),
 ('aed6psww', 11.091345964879876),
 ('4fwydrtj', 10.971627026912595),
 ('20evfmcf', 10.792959313637834),
 ('hrfbygub', 10.71277964354996),
 ('btiy6jhr', 10.70389371228511),
 ('zpcdjsti', 10.69985904263975),
 ('buchzofy', 10.69985904263975),
 ('g12i1jig', 10.676981362115992),
 ('qb6fea06', 10.660131203176956),
 ('kkga96h9', 10.660021022075943),
 ('9611eglg', 10.6580210613391),
 ('pknaxvhg', 10.591796029